### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("ollama:llama3.2:latest")
response=llm.invoke("What is the weather in New York?")
response.content

"I'm not able to provide real-time or current weather conditions. However, I can suggest some ways for you to find out the current weather in New York.\n\n1. Check online weather websites: You can check websites like weather.com, accuweather.com, or wunderground.com for the current weather conditions and forecast in New York.\n2. Use a mobile app: You can download mobile apps like Dark Sky or Weather Underground to get real-time weather updates on your smartphone.\n3. Tune into local news: You can watch local news or listen to radio stations that provide weather updates for New York.\n\nPlease note that the weather in New York can vary greatly depending on the time of year, location, and other factors. If you're looking for specific information on a particular area in New York, I'd be happy to try and help you find it."

In [4]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the weather in a location"""
    return f"The weather in {location} is sunny."


llm_with_tool=llm.bind_tools([get_weather])
response=llm_with_tool.invoke("What is the weather in New York?")
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-27T14:36:22.002018Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1317471400, 'load_duration': 317944200, 'prompt_eval_count': 150, 'prompt_eval_duration': 97760900, 'eval_count': 18, 'eval_duration': 881473200, 'model_name': 'llama3.2:latest'}, id='lc_run--019dcf5e-a949-7fa1-85db-8f8c7b7625bf-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '9530f6d9-4c19-43f2-8dca-e57277d99d39', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'total_tokens': 168})

In [5]:
for tool_call in response.tool_calls:
    print(tool_call)

{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '9530f6d9-4c19-43f2-8dca-e57277d99d39', 'type': 'tool_call'}


### Tool Execution Loop

In [6]:
messages=[{"role":"user","content":"What is the weather in New York?"}]
ai_msg=llm_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response=llm_with_tool.invoke(messages)
final_response

AIMessage(content="Based on the output from the API, the current weather condition in New York is Sunny. However, I don't have real-time access to current weather conditions. If you want to know the current weather, I recommend checking a weather website or app for the most up-to-date information. Would you like me to help with anything else?", additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-27T14:38:33.2715076Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4803311600, 'load_duration': 318477100, 'prompt_eval_count': 100, 'prompt_eval_duration': 792473300, 'eval_count': 68, 'eval_duration': 3600425700, 'model_name': 'llama3.2:latest'}, id='lc_run--019dcf60-9c73-7221-be7b-13235aba2b7c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 100, 'output_tokens': 68, 'total_tokens': 168})

In [7]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-27T14:38:28.4643398Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1301568100, 'load_duration': 357688600, 'prompt_eval_count': 150, 'prompt_eval_duration': 104341000, 'eval_count': 18, 'eval_duration': 818662900, 'model_name': 'llama3.2:latest'}, id='lc_run--019dcf60-9755-7f43-ae28-1010a11d057f-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '6d01cecf-bd5e-4217-9a01-e4c469e57775', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'total_tokens': 168}),
 ToolMessage(content='The weather in New York is sunny.', name='get_weather', tool_call_id='6d01cecf-bd5e-4217-9a01-e4c469e57775')]